# SALITAyo Tagalog Active-Voice Lexical Simplifier

This Colab notebook trains a **candidate** Tagalog model for your app.

Goal:
- Keep the same meaning and facts.
- Do not hallucinate.
- Do not add information.
- Do not delete information.
- Replace difficult Tagalog/Filipino words with common words.
- Convert passive/wordy Tagalog sentences into clearer active voice when possible.
- Preserve names, places, organizations, acronyms, numbers, and dates.

Important: this notebook saves a **candidate model** only. Do not delete your app model until the tests pass.

Base model: `google/mt5-base` recommended. Use `google/mt5-small` if Colab runs out of memory.

## Fixed Version Notes

This version fixes the Colab issues you hit:
- no infinite active-pair generation loop
- uses `processing_class=tokenizer` instead of removed `tokenizer=` trainer argument
- masks label padding as `-100`
- disables fp16 to avoid `nan` validation loss on Colab T4

Upload this fresh notebook and run from the top in a restarted runtime.
- adds broader `Ang X ay Y` restructuring examples so non-passive Tagalog sentences do not simply copy


In [ ]:
!pip -q install -U transformers datasets accelerate sentencepiece pandas scikit-learn

import os
import re
import json
import random
import shutil
import pandas as pd
import numpy as np
import torch

from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Optional: Mount Google Drive
Use Drive so the trained candidate model is saved after Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/salitayo_tagalog_active_voice'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Drive output:', DRIVE_OUT)


## Configuration

This trains with the same prefix your backend uses for Tagalog:

`restructure fil:`

That matters because your app's `restructurer_inference.py` sends Tagalog input using that prefix.

In [ ]:
BASE_MODEL = 'google/mt5-base'
# BASE_MODEL = 'google/mt5-small'  # use this if T4 memory is not enough

OUTPUT_DIR = '/content/salitayo_tagalog_candidate_training'
FINAL_DIR = '/content/salitayo_tagalog_candidate_model'

TASK_PREFIX = 'restructure fil: '
MAX_INPUT_LENGTH = 192
MAX_TARGET_LENGTH = 192
EPOCHS = 5
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-5

# Optional: add your own checked CSV later. Expected columns: complex,simple OR original,restructured
USER_DATA_PATH = ''

# Public external support datasets. They are not direct simplification pairs, but they provide real Filipino text.
USE_EXTERNAL_SUPPORT_DATA = True
EXTERNAL_FILIPINO_DATASETS = [
    'josephimperial/filipino_elementary_periodical_exams',
    'jfernandez/cebuano-filipino-sentences',
]


## Tagalog Lexical Replacement Dictionary

This is **not the final app logic**. It is used to create controlled training examples.

The model learns the pattern from many examples:
`hard/wordy Tagalog -> common Tagalog`, while keeping the same information.

In [ ]:
HARD_TO_COMMON = {
    'akumulasyon': 'pag-ipon',
    'analisis': 'pagsusuri',
    'asistensya': 'tulong',
    'benepisyo': 'pakinyabang',
    'departamento': 'tanggapan',
    'detalyado': 'may detalye',
    'epektibo': 'mabisa',
    'ekstensibo': 'malawak',
    'eliminahin': 'alisin',
    'emosyonal': 'damdamin',
    'estratehiya': 'paraan',
    'evaluasyon': 'pagsusuri',
    'gumaganap': 'gumagawa',
    'identipikasyon': 'pagkilala',
    'implementasyon': 'pagsasagawa',
    'impormasyon': 'kaalaman',
    'inisyal': 'una',
    'interaksyon': 'pakikipag-ugnayan',
    'interpretasyon': 'pag-unawa',
    'isinasagawa': 'ginagawa',
    'isinagawa': 'ginawa',
    'kinakailangan': 'kailangan',
    'kognitibo': 'pag-iisip',
    'kolaborasyon': 'pagtutulungan',
    'komplikado': 'mahirap',
    'komprehensibo': 'kumpleto',
    'komunikasyon': 'pakikipag-usap',
    'konklusyon': 'wakas',
    'konsekwensya': 'bunga',
    'konsiderable': 'malaki',
    'konstruksyon': 'pagtatayo',
    'kontemporaryo': 'makabago',
    'konteksto': 'kalagayan',
    'mahalagang': 'importanteng',
    'makabuluhan': 'mahalaga',
    'malawakang': 'malawak',
    'maramihang': 'marami',
    'metodolohiya': 'paraan',
    'obserbasyon': 'pagmamasid',
    'organisasyon': 'samahan',
    'partisipasyon': 'pagsali',
    'perspektibo': 'pananaw',
    'potensyal': 'posible',
    'prayoridad': 'una',
    'presentasyon': 'paglalahad',
    'proseso': 'hakbang',
    'proyekto': 'gawain',
    'rekomendasyon': 'mungkahi',
    'representasyon': 'paglalarawan',
    'responsibilidad': 'tungkulin',
    'resulta': 'kinalabasan',
    'signipikante': 'mahalaga',
    'sitwasyon': 'kalagayan',
    'solusyon': 'lunas',
    'teknolohiya': 'makinarya',
    'terminolohiya': 'mga salita',
}

PROTECTED_PATTERN = re.compile(r'\b(?:[A-Z]{2,}|[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+|\d+(?:\.\d+)?%?)\b')

def match_case(source, replacement):
    if source.isupper():
        return replacement.upper()
    if source[:1].isupper():
        return replacement[:1].upper() + replacement[1:]
    return replacement

print('Dictionary entries:', len(HARD_TO_COMMON))


## Controlled Active-Voice Training Pairs

These pairs teach active voice directly:

Passive/wordy:
`Ang datos ay sinuri ng mga mananaliksik.`

Active:
`Sinuri ng mga mananaliksik ang datos.`

In [ ]:
ACTORS = [
    'guro', 'mag-aaral', 'mga mag-aaral', 'mananaliksik', 'mga mananaliksik',
    'doktor', 'tagapagsalita', 'pangkat', 'komunidad', 'paaralan', 'SALITAyo'
]

OBJECTS = [
    'datos', 'ulat', 'aralin', 'impormasyon', 'proseso', 'proyekto', 'gawain',
    'tanong', 'sagot', 'plano', 'resulta', 'dokumento', 'kuwento', 'pananaliksik'
]

PASSIVE_ACTIVE = [
    ('sinuri', 'Sinuri'),
    ('ipinaliwanag', 'Ipinaliwanag'),
    ('isinulat', 'Isinulat'),
    ('binasa', 'Binasa'),
    ('ginawa', 'Ginawa'),
    ('sinimulan', 'Sinimulan'),
    ('tinapos', 'Tinapos'),
    ('ipinakita', 'Ipinakita'),
    ('inayos', 'Inayos'),
    ('tinulungan', 'Tinulungan'),
]

WORDY_PATTERNS = [
    ('Ang partisipasyon ng {actor} ay kinakailangan sa {obj}.', 'Kailangang sumali ang {actor} sa {obj}.'),
    ('Ang implementasyon ng {obj} ay isinagawa ng {actor}.', 'Ginawa ng {actor} ang {obj}.'),
    ('Ang komunikasyon ng {actor} ay mahalaga sa {obj}.', 'Mahalaga ang pakikipag-usap ng {actor} sa {obj}.'),
    ('Ang evaluasyon ng {obj} ay isinagawa ng {actor}.', 'Sinuri ng {actor} ang {obj}.'),
    ('Ang komplikadong proseso ay ipinaliwanag ng {actor}.', 'Ipinaliwanag ng {actor} ang mahirap na hakbang.'),
    ('Ang impormasyon tungkol sa {obj} ay ipinakita ng {actor}.', 'Ipinakita ng {actor} ang kaalaman tungkol sa {obj}.'),
    ('Ang rekomendasyon para sa {obj} ay ibinigay ng {actor}.', 'Nagbigay ng mungkahi ang {actor} para sa {obj}.'),
    ('Ang metodolohiya ng {obj} ay ipinaliwanag ng {actor}.', 'Ipinaliwanag ng {actor} ang paraan ng {obj}.'),
]


def lexical_replace_tl(text):
    protected = [m.span() for m in PROTECTED_PATTERN.finditer(text)]
    def blocked(start, end):
        return any(start < e and end > s for s, e in protected)

    replacements = []
    for hard, common in sorted(HARD_TO_COMMON.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = re.compile(rf'\b{re.escape(hard)}\b', flags=re.IGNORECASE)
        for m in pattern.finditer(text):
            if not blocked(m.start(), m.end()):
                replacements.append((m.start(), m.end(), match_case(m.group(0), common), m.group(0)))

    out = text
    used = []
    occupied = []
    for start, end, common, original in sorted(replacements, reverse=True):
        if any(start < e and end > s for s, e in occupied):
            continue
        out = out[:start] + common + out[end:]
        used.append((original, common))
        occupied.append((start, end))
    return out, used


def generate_active_voice_pairs(target_count=7000):
    pairs = []
    seen = set()

    for actor in ACTORS:
        for obj in OBJECTS:
            for passive, active in PASSIVE_ACTIVE:
                original = f'Ang {obj} ay {passive} ng {actor}.'
                target = f'{active} ng {actor} ang {obj}.'
                key = (original, target)
                if key not in seen:
                    pairs.append({'original': original, 'restructured': target, 'source': 'active_voice_template'})
                    seen.add(key)

    attempts = 0
    while len(pairs) < target_count and attempts < target_count * 20:
        attempts += 1
        actor = random.choice(ACTORS)
        obj = random.choice(OBJECTS)
        pattern, target_pattern = random.choice(WORDY_PATTERNS)
        original = pattern.format(actor=actor, obj=obj)
        target = target_pattern.format(actor=actor, obj=obj)
        key = (original, target)
        if key not in seen:
            pairs.append({'original': original, 'restructured': target, 'source': 'wordy_template'})
            seen.add(key)

    return pairs

active_pairs = generate_active_voice_pairs(2000)
print('Active voice pairs:', len(active_pairs))
for row in active_pairs[:5]:
    print('\nORIGINAL:', row['original'])
    print('TARGET  :', row['restructured'])


## Broader `Ang X ay Y` Restructuring Pairs

This section fixes the copying problem for Tagalog sentences that are not passive voice. It teaches the model broader `ay` restructuring patterns, such as:

- `Ang tunay na asenso ay walang nakikitang sulok.` -> `Walang nakikitang sulok ang tunay na pag-unlad.`
- `Ang pangunahing layunin ay malinaw.` -> `Malinaw ang pangunahing layunin.`
- `Ang mahirap na proseso ay mahalaga.` -> `Mahalaga ang mahirap na proseso.`

These are still controlled pairs: same information, no bullets, no added facts.


In [ ]:
ABSTRACT_SUBJECTS = [
    ('asenso', 'pag-unlad'),
    ('tagumpay', 'tagumpay'),
    ('pagkatuto', 'pagkatuto'),
    ('komunikasyon', 'pakikipag-usap'),
    ('partisipasyon', 'pagsali'),
    ('implementasyon', 'pagsasagawa'),
    ('edukasyon', 'pag-aaral'),
    ('pananaliksik', 'pananaliksik'),
    ('proyekto', 'gawain'),
    ('proseso', 'hakbang'),
    ('solusyon', 'lunas'),
    ('rekomendasyon', 'mungkahi'),
    ('impormasyon', 'kaalaman'),
    ('metodolohiya', 'paraan'),
]

MODIFIERS = ['', 'tunay na ', 'mahalagang ', 'pangunahing ', 'bagong ', 'mahirap na ', 'komplikadong ']
PREDICATES = [
    'mahalaga',
    'malinaw',
    'kailangan',
    'makabuluhan',
    'mahirap unawain',
    'madaling sundin',
    'kapaki-pakinabang',
    'bahagi ng pag-unlad',
    'paraan ng pagtutulungan',
]
WALANG_PHRASES = [
    'walang nakikitang sulok',
    'walang malinaw na hangganan',
    'walang madaling daan',
    'walang iisang paraan',
]
HINDI_PHRASES = [
    'hindi laging madali',
    'hindi agad nakikita',
    'hindi dapat minamadali',
    'hindi pare-pareho sa lahat',
]


def cap_first(text):
    text = text.strip()
    return text[:1].upper() + text[1:] if text else text


def generate_ay_restructure_pairs(target_count=3500):
    pairs = []
    seen = set()

    def add(original, target, source='ay_restructure_template'):
        if original == target:
            return
        key = (original, target)
        if key not in seen:
            pairs.append({'original': original, 'restructured': target, 'source': source})
            seen.add(key)

    for hard_subject, simple_subject in ABSTRACT_SUBJECTS:
        for modifier in MODIFIERS:
            subj = f'{modifier}{hard_subject}'.strip()
            simple_subj = f'{modifier}{simple_subject}'.strip()

            for pred in PREDICATES:
                add(f'Ang {subj} ay {pred}.', f'{cap_first(pred)} ang {simple_subj}.')

            for phrase in WALANG_PHRASES:
                add(f'Ang {subj} ay {phrase}.', f'{cap_first(phrase)} ang {simple_subj}.')

            for phrase in HINDI_PHRASES:
                add(f'Ang {subj} ay {phrase}.', f'{cap_first(phrase)} ang {simple_subj}.')

    # Add varied object/location forms so it generalizes beyond abstract nouns.
    extra_subjects = [
        'aralin', 'ulat', 'gawain', 'pagsasanay', 'paliwanag', 'panuto',
        'desisyon', 'plano', 'layunin', 'resulta', 'tanong', 'sagot'
    ]
    extra_predicates = [
        'maikli', 'mahaba', 'malinaw', 'mahirap', 'madali', 'mahalaga',
        'kailangan sa klase', 'kapaki-pakinabang sa mag-aaral', 'bahagi ng aralin'
    ]
    for subj in extra_subjects:
        for modifier in MODIFIERS:
            full_subj = f'{modifier}{subj}'.strip()
            for pred in extra_predicates:
                add(f'Ang {full_subj} ay {pred}.', f'{cap_first(pred)} ang {full_subj}.')

    random.shuffle(pairs)
    return pairs[:target_count]

ay_restructure_pairs = generate_ay_restructure_pairs(3500)
print('AY restructure pairs:', len(ay_restructure_pairs))
for row in ay_restructure_pairs[:8]:
    print('\nORIGINAL:', row['original'])
    print('TARGET  :', row['restructured'])


## External Filipino Support Data

I checked Hugging Face for Tagalog/Filipino resources. There is no large public dataset that directly matches your task: `complex Tagalog -> active voice simplified Tagalog`.

So this notebook uses reliable public Filipino text as support data:
- `josephimperial/filipino_elementary_periodical_exams`: elementary Filipino questions from DepEd-style exams.
- `jfernandez/cebuano-filipino-sentences`: 105k Cebuano-Filipino sentence pairs; we use the Filipino side as natural sentence support.

These are not simplification labels by themselves. The notebook only turns them into pairs when controlled lexical replacement is possible.

In [ ]:
def extract_filipino_sentences_from_external(max_sentences=8000):
    sentences = []

    if not USE_EXTERNAL_SUPPORT_DATA:
        return sentences

    try:
        exams = load_dataset('josephimperial/filipino_elementary_periodical_exams', split='train')
        for row in exams:
            q = str(row.get('question', '')).strip()
            if q:
                sentences.append(q)
            for opt in row.get('options', []) or []:
                opt = str(opt).strip()
                if len(opt.split()) >= 4:
                    sentences.append(opt)
    except Exception as exc:
        print('Could not load elementary exams:', exc)

    try:
        cf = load_dataset('jfernandez/cebuano-filipino-sentences', split='train')
        for row in cf.select(range(min(len(cf), max_sentences))):
            pair = row.get('set', [])
            if isinstance(pair, list) and pair:
                fil = str(pair[0]).strip()  # first side is Filipino in sampled rows
                if 4 <= len(fil.split()) <= 25:
                    sentences.append(fil)
    except Exception as exc:
        print('Could not load Cebuano-Filipino sentences:', exc)

    clean = []
    seen = set()
    for s in sentences:
        s = re.sub(r'\s+', ' ', s).strip()
        if s and s not in seen and len(s.split()) >= 4:
            clean.append(s)
            seen.add(s)
    return clean[:max_sentences]

external_sentences = extract_filipino_sentences_from_external()
print('External Filipino support sentences:', len(external_sentences))
for s in external_sentences[:10]:
    print('-', s)

external_pairs = []
seen = set()
for sentence in external_sentences:
    rewritten, used = lexical_replace_tl(sentence)
    if used and rewritten != sentence:
        key = (sentence, rewritten)
        if key not in seen:
            external_pairs.append({'original': sentence, 'restructured': rewritten, 'source': 'external_lexical'})
            seen.add(key)

print('External lexical pairs:', len(external_pairs))
for row in external_pairs[:5]:
    print('\nORIGINAL:', row['original'])
    print('TARGET  :', row['restructured'])


## Additional Synthetic Lexical Pairs
These cover more hard-word replacements in Tagalog contexts.

In [ ]:
LEXICAL_TEMPLATES = [
    'Ang {hard} ay mahalaga sa {obj}.',
    'Ginamit ng {actor} ang {hard} sa {obj}.',
    'Ipinaliwanag ng {actor} ang {hard} tungkol sa {obj}.',
    'Kailangan ng {actor} ang {hard} para sa {obj}.',
    'Ang {obj} ay may {hard} na bahagi.',
    'Tinalakay ng {actor} ang {hard} sa klase.',
    'Nakita ng {actor} ang {hard} sa ulat.',
]


def generate_lexical_pairs(target_count=5000):
    pairs = []
    seen = set()
    hard_words = list(HARD_TO_COMMON.keys())
    while len(pairs) < target_count:
        hard = random.choice(hard_words)
        actor = random.choice(ACTORS)
        obj = random.choice(OBJECTS)
        template = random.choice(LEXICAL_TEMPLATES)
        original = template.format(hard=hard, actor=actor, obj=obj)
        target, used = lexical_replace_tl(original)
        if not used or original == target:
            continue
        key = (original, target)
        if key not in seen:
            pairs.append({'original': original, 'restructured': target, 'source': 'lexical_template'})
            seen.add(key)
    return pairs

lexical_pairs = generate_lexical_pairs(5000)
print('Lexical pairs:', len(lexical_pairs))
for row in lexical_pairs[:5]:
    print('\nORIGINAL:', row['original'])
    print('TARGET  :', row['restructured'])


## Optional: Load Your Own Checked Pairs

Use this later if you have a CSV. Do **not** use old bullet-style outputs if you want active voice and clean sentence output.

In [ ]:
def load_user_pairs(path):
    if not path:
        return []
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    if path.endswith('.csv'):
        df = pd.read_csv(path)
    elif path.endswith('.jsonl'):
        df = pd.read_json(path, lines=True)
    elif path.endswith('.json'):
        df = pd.read_json(path)
    else:
        raise ValueError('Use CSV, JSON, or JSONL.')

    if {'complex', 'simple'}.issubset(df.columns):
        df = df.rename(columns={'complex': 'original', 'simple': 'restructured'})
    elif not {'original', 'restructured'}.issubset(df.columns):
        raise ValueError('Dataset must have complex/simple or original/restructured columns.')

    rows = []
    for _, row in df[['original', 'restructured']].dropna().iterrows():
        original = str(row['original']).strip()
        target = str(row['restructured']).strip()
        if original and target and original != target and '?' not in target:
            rows.append({'original': original, 'restructured': target, 'source': 'user'})
    return rows

user_pairs = load_user_pairs(USER_DATA_PATH)
print('User checked pairs:', len(user_pairs))


## Build and Filter Dataset

In [ ]:
def normalize_words(text):
    return re.findall(r"[A-Za-z?-?0-9'\-]+", str(text).lower())

def protected_terms(text):
    return set(m.group(0) for m in PROTECTED_PATTERN.finditer(str(text)))

def pair_is_safe(original, target):
    if not original or not target or original.strip() == target.strip():
        return False
    if '?' in target or '\n' in target:
        return False
    o_words = normalize_words(original)
    t_words = normalize_words(target)
    if not o_words or not t_words:
        return False
    ratio = len(t_words) / max(len(o_words), 1)
    if ratio < 0.55 or ratio > 1.45:
        return False
    for term in protected_terms(original):
        if term not in target:
            return False
    return True

all_pairs = active_pairs + ay_restructure_pairs + lexical_pairs + external_pairs + user_pairs
df = pd.DataFrame(all_pairs)[['original', 'restructured', 'source']].drop_duplicates().reset_index(drop=True)
before = len(df)
df = df[df.apply(lambda r: pair_is_safe(r['original'], r['restructured']), axis=1)].reset_index(drop=True)
print('Before filter:', before)
print('After filter:', len(df))
print(df['source'].value_counts())

train_df, eval_df = train_test_split(df, test_size=0.08, random_state=SEED, stratify=df['source'] if df['source'].nunique() > 1 else None)
print('Train:', len(train_df), 'Eval:', len(eval_df))
df.head()


## Train mT5

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_ds = Dataset.from_pandas(eval_df.reset_index(drop=True))

def preprocess(batch):
    inputs = [TASK_PREFIX + x for x in batch['original']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch['restructured'], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs['labels'] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels['input_ids']
    ]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_tok = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    fp16=False,  # safer on Colab; avoids nan loss on some T4/transformers combos
    logging_steps=50,
    save_total_limit=2,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
    data_collator=collator,
)

trainer.train()


## Acceptance Tests Before Export
Do not replace your app model unless this section looks good.

In [ ]:
def simplify_tl(text, max_new_tokens=160):
    prompt = TASK_PREFIX + text.strip()
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LENGTH).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            length_penalty=0.95,
            early_stopping=True,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

GOLD_TESTS = [
    'Ang tunay na asenso ay walang nakikitang sulok.',
    'Ang datos ay sinuri ng mga mananaliksik.',
    'Ang implementasyon ng proyekto ay isinagawa ng guro.',
    'Ang komplikadong proseso ay ipinaliwanag ng doktor.',
    'Ang partisipasyon ng mag-aaral ay kinakailangan sa gawain.',
    'Ang komunikasyon sa klase ay mahalaga para sa mga mag-aaral.',
    'Ang impormasyon tungkol sa aralin ay ipinakita ng tagapagsalita.',
    'Ang metodolohiya ng pananaliksik ay ipinaliwanag ng pangkat.',
    'Ang rekomendasyon para sa proyekto ay ibinigay ng komunidad.',
]

for text in GOLD_TESTS:
    pred = simplify_tl(text)
    print('\nINPUT :', text)
    print('OUTPUT:', pred)
    print('COPY? :', pred.strip().lower() == text.strip().lower())
    print('BULLET:', '?' in pred)


## Save Candidate Model
This does not touch your current app model.

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

with open(os.path.join(FINAL_DIR, 'salitayo_tagalog_training_config.json'), 'w') as f:
    json.dump({
        'task': 'tagalog_active_voice_lexical_simplification',
        'base_model': BASE_MODEL,
        'task_prefix': TASK_PREFIX,
        'pairs': len(df),
        'sources': df['source'].value_counts().to_dict(),
    }, f, indent=2)

if 'DRIVE_OUT' in globals():
    drive_model_dir = os.path.join(DRIVE_OUT, 'salitayo_tagalog_candidate_model')
    if os.path.exists(drive_model_dir):
        shutil.rmtree(drive_model_dir)
    shutil.copytree(FINAL_DIR, drive_model_dir)
    print('Saved to Drive:', drive_model_dir)

shutil.make_archive('/content/salitayo_tagalog_candidate_model', 'zip', FINAL_DIR)
print('Candidate model:', FINAL_DIR)
print('Zip:', '/content/salitayo_tagalog_candidate_model.zip')


## How to Use Safely

Only after the acceptance tests look good:

1. Keep your old app model as backup.
2. Copy `salitayo_tagalog_candidate_model` into your project only when ready.
3. For a two-model setup later, use this candidate for Tagalog and keep your current model for English.

Expected Colab T4 time:
- `google/mt5-small`: about 45 minutes to 1.5 hours.
- `google/mt5-base`: about 2 to 4 hours.

The dataset is enough for a strong first candidate because it creates around 12k controlled pairs plus external Filipino support pairs when available. It is not a perfect human-labeled simplification corpus, because a large public Tagalog active-voice simplification dataset does not appear to exist.